# Namma Space — End-to-End 3D Reconstruction Training Pipeline
### Techfest, IIT Bombay — Spatial Engine

This notebook trains a **3D Gaussian Splatting (3DGS)** digital twin from a standard smartphone video.

**Pipeline Flow:**
1. GPU & CUDA Environment Setup
2. COLMAP & Nerfstudio (`splatfacto`) Installation
3. Input Video Ingestion (Upload or Google Drive)
4. Automated Structure-from-Motion (COLMAP pose estimation & sparse reconstruction)
5. 3D Gaussian Splatting Model Training (`splatfacto`)
6. Web-Ready 3D Model Export (`.ply` / `.splat`)

## Step 1: Verify Hardware Accelerator
Make sure **Runtime > Change runtime type > Hardware accelerator** is set to **T4 GPU** or **A100 GPU**.

In [ ]:
!nvidia-smi

## Step 2: Install System Dependencies (COLMAP, FFmpeg, Ninja)

In [ ]:
# Install COLMAP and build tools
!apt-get update -y && apt-get install -y colmap ffmpeg ninja-build

## Step 3: Install Nerfstudio & Gaussian Splatting Engine
Nerfstudio's `splatfacto` is the industry standard for fast, high-quality 3D Gaussian Splatting.

In [ ]:
!pip install --upgrade pip
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install nerfstudio

## Step 4: Mount Google Drive or Upload Video Capture
You can either mount Google Drive (recommended for large 4K smartphone videos) or upload directly.

In [ ]:
import os
from google.colab import drive

# Mount Drive (optional, uncomment if using Drive)
# drive.mount('/content/drive')

# Workspace directory setup
WORKSPACE_DIR = "/content/namma_space"
DATA_DIR = os.path.join(WORKSPACE_DIR, "data")
OUTPUT_DIR = os.path.join(WORKSPACE_DIR, "outputs")
EXPORT_DIR = os.path.join(WORKSPACE_DIR, "exports")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(EXPORT_DIR, exist_ok=True)

print(f"Workspace initialized at {WORKSPACE_DIR}")

### Step 4b: Provide Your Smartphone Video
Place your input video path below. 
- If uploaded to Colab directly: `INPUT_VIDEO = "/content/room_capture.mp4"`
- Or upload using the Colab file upload widget below:

In [ ]:
from google.colab import files

# If you haven't uploaded your video yet, run this cell to upload directly:
uploaded = files.upload()
video_filename = list(uploaded.keys())[0]
INPUT_VIDEO = os.path.abspath(video_filename)
print(f"Using input video: {INPUT_VIDEO}")

## Step 5: End-to-End Structure-from-Motion (COLMAP via `ns-process-data`)
This step:
- Extracts optimal keyframes from your video (at 2-3 fps)
- Runs feature matching via COLMAP
- Reconstructs camera positions and sparse geometry

In [ ]:
PROCESSED_DATA_DIR = os.path.join(DATA_DIR, "processed_room")

# Automatically run frame extraction and COLMAP feature matching
!ns-process-data video \
    --data "{INPUT_VIDEO}" \
    --output-dir "{PROCESSED_DATA_DIR}" \
    --num-frames-target 250 \
    --matching-method exhaustive

## Step 6: Train the 3D Gaussian Splatting Model (`splatfacto`)
This trains the 3D Gaussian representations in spatial coordinates with full photometric rendering.

In [ ]:
# Train splatfacto model (Gaussian Splatting)
# 15,000 - 30,000 steps produces high-fidelity photorealistic indoor twins
!ns-train splatfacto \
    --data "{PROCESSED_DATA_DIR}" \
    --output-dir "{OUTPUT_DIR}" \
    --max-num-iterations 20000 \
    --pipeline.model.cull-alpha-thresh 0.005 \
    --viewer.quit-on-train-completion True

## Step 7: Export Web-Ready 3D Digital Twin (`.ply` / `.splat`)
Exports the trained Gaussian splats into a file ready to be loaded by Three.js in your Web Viewer.

In [ ]:
import glob

# Locate the generated config.yml from training
config_files = glob.glob(f"{OUTPUT_DIR}/**/config.yml", recursive=True)
if not config_files:
    raise FileNotFoundError("Training configuration file not found!")

latest_config = max(config_files, key=os.path.getctime)
print(f"Found latest training config: {latest_config}")

# Export to standard Gaussian Splat PLY
!ns-export gaussian-splat \
    --load-config "{latest_config}" \
    --output-dir "{EXPORT_DIR}"

exported_plys = glob.glob(f"{EXPORT_DIR}/*.ply")
print(f"Successfully exported web model: {exported_plys}")

## Step 8: Download Exported 3D Model
Download the generated `.ply` or `.splat` file to your laptop to place in your Web Viewer `public/` directory.

In [ ]:
from google.colab import files
for ply in exported_plys:
    files.download(ply)